**Install Bio package**

In [ ]:
pip install Bio

**Run GRAVY**
1. Click on the Folder icon.
2. Click on 'Upload to session storage'.
3. Upload your FASTA file - named Input.fasta
4. Run the next cell.

In [ ]:
# GRAVY score calculation and FASTA parsing
from Bio import SeqIO
import numpy as np
import pandas as pd

# Hydropathy scale based on Kyte-Doolittle
hydropathy_scale = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8,
    'P': -1.6, 'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

# Function to calculate GRAVY score for a sequence
def calculate_gravy(sequence):
    # Only consider amino acids in the hydropathy scale
    hydropathy_values = [hydropathy_scale.get(aa, 0) for aa in sequence]
    return np.mean(hydropathy_values)

# Read the FASTA file and calculate GRAVY scores
def process_fasta(file_path):
    results = []

    # Parse the FASTA file
    for record in SeqIO.parse(file_path, "fasta"):
        # Extract ID from the header (after the first pipe character)
        record_id = record.id.split("|")[1]
        sequence = str(record.seq)

        # Calculate GRAVY score
        gravy_score = calculate_gravy(sequence)

        # Store the results
        results.append([record_id, gravy_score])

    # Create a DataFrame for easy saving to .tsv
    df = pd.DataFrame(results, columns=["ID", "GRAVY_Score"])

    # Save results as a tsv file
    output_file = "gravy_scores.tsv"
    df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")

# Use this function by passing the file path to Input.fasta
file_path = "Input.fasta"  # Modify if the file path is different
process_fasta(file_path)


**DTMHMM Annotator**

If you have the predicted_topologies.3line and the TMRs.gff3 files from a DTMHMM run for your FASTA file, you can use the following cells to add more annotations to the output from the previous cell. Just upload the predicted_topologies.3line and the TMRs.gff3 files into the root of the session storage and run the following cells in sequence.

In [ ]:

# Import libraries
import pandas as pd

# Set file paths
# Change these paths if your files are elsewhere
gravy_file = "gravy_scores.tsv"
dtmhmm_file = "predicted_topologies.3line"
output_file = "GRAVY_annotated.tsv"

# Load GRAVY file
gravy_df = pd.read_csv(gravy_file, sep="\t")
print("GRAVY file loaded. Sample:")
print(gravy_df.head())

# Parse DeepTMHMM predictions

dtmhmm_dict = {}

with open(dtmhmm_file, 'r') as f:
    for line in f:
        if line.startswith(">"):
            # Split header on '|' first
            parts = line.strip().split("|")
            if len(parts) >= 3:
                protein_id = parts[1].strip()  # still use the second part as ID
                # Take the last part after '|', then split by space to get only the prediction
                prediction = parts[-1].strip().split()[0]
                dtmhmm_dict[protein_id] = prediction

print("DeepTMHMM predictions loaded for", len(dtmhmm_dict), "proteins.")

# Add DTMHMM column to GRAVY dataframe
gravy_df['DTMHMM'] = gravy_df['ID'].map(dtmhmm_dict)

# Save annotated TSV
gravy_df.to_csv(output_file, sep="\t", index=False)
print(f"Annotated file saved as {output_file}.")


**More annotations**

In [ ]:
# Add Length, TMRs, and residue-region annotations

import pandas as pd

# File paths
annotated_file = "GRAVY_annotated.tsv"  # previously annotated file
gff_file = "TMRs.gff3"
output_file = "GRAVY_annotated_final.tsv"

# Load previously annotated GRAVY file
gravy_df = pd.read_csv(annotated_file, sep="\t")
print("Loaded previously annotated GRAVY file. Sample:")
print(gravy_df.head())

# --- STEP 2: Parse GFF3 file ---
# Dictionary: ID -> {'Length': int, 'TMRs': int, 'Regions': [str, str, ...]}
gff_dict = {}

with open(gff_file, 'r') as f:
    current_id = None
    for line in f:
        line = line.strip()
        if line.startswith('##') or line == '':
            continue  # skip file header or empty lines
        elif line.startswith('#'):
            # Comment lines with length and TMRs
            if 'Length:' in line:
                parts = line.split('|')
                current_id = parts[1].strip()  # second pipe field
                length = int(line.split('Length:')[1].strip())
                gff_dict[current_id] = {'Length': length, 'TMRs': None, 'Regions': []}
            elif 'Number of predicted TMRs:' in line:
                tmr_count = int(line.split('Number of predicted TMRs:')[1].strip())
                gff_dict[current_id]['TMRs'] = tmr_count
        elif line.startswith('//'):
            current_id = None  # end of sequence block
        else:
            # Residue region line
            # Split only on the first tab or whitespace
            if '\t' in line:
                header, region_info = line.split('\t', 1)
            else:
                header, region_info = line.split(None, 1)
            region_id = header.split('|')[1].strip()  # second pipe field
            gff_dict[region_id]['Regions'].append(region_info.strip())

print(f"Parsed GFF3 file for {len(gff_dict)} proteins.")

# --- STEP 3: Add Length and TMRs to GRAVY dataframe ---
gravy_df['Length'] = gravy_df['ID'].map(lambda x: gff_dict[x]['Length'] if x in gff_dict else None)
gravy_df['TMRs'] = gravy_df['ID'].map(lambda x: gff_dict[x]['TMRs'] if x in gff_dict else None)


# --- STEP 4: Add region annotations as separate columns with dash ---

# Find maximum number of regions
max_regions = max(len(v['Regions']) for v in gff_dict.values())
print(f"Maximum number of residue regions: {max_regions}")

for i in range(max_regions):
    col_name = f'Region_{i+1}'
    def format_region(x, i=i):
        if x in gff_dict and i < len(gff_dict[x]['Regions']):
            region_parts = gff_dict[x]['Regions'][i].split()
            if len(region_parts) == 3:
                return f"{region_parts[0]} {region_parts[1]}-{region_parts[2]}"
            else:
                return gff_dict[x]['Regions'][i]  # leave as-is if not 3 parts
        else:
            return None
    gravy_df[col_name] = gravy_df['ID'].map(format_region)

# --- STEP 5: Save updated file ---
gravy_df.to_csv(output_file, sep="\t", index=False)
print(f"Annotated file saved as {output_file}.")
